In [1]:
import os
import json
from tqdm.auto import tqdm
from pathlib import Path

import pandas as pd

from concurrent.futures import ThreadPoolExecutor, as_completed

In [2]:
# RUN ONCE: change working directory to project root
cwd = Path.cwd()

pwd = cwd.parent

os.chdir(pwd)
print(f"Changed working directory to {pwd}")

if Path.cwd() != pwd:
    raise RuntimeError(f"Failed to change working directory to {pwd}")

Changed working directory to /Users/jdk/projects/recruiting-reader


In [3]:
# create driver
from src.scraper.driver import make_driver
driver = make_driver(headless=True)

In [ ]:
# driver.quit()

## Load 2025 Links & Data Cache

In [3]:
with open("data/urls_2025_transfers.json", "r") as f:
    urls_2025 = json.load(f)

In [4]:
portal_df_test = pd.read_csv("data/portal_2025_transfers_1218_run5.csv")
portal_df_test['transfer_destination'] = portal_df_test['transfer_destination'].str.replace('amp;', '', regex=False)
portal_df_test['transfer_origin'] = portal_df_test['transfer_origin'].str.replace('amp;', '', regex=False)

## Investigate Missing Values

In [ ]:
portal_df_test.isna().sum()

id_247                     0
name                       0
pos_247                    0
hs_name                    5
hs_city                    0
hs_state                   0
transfer_rating            0
transfer_year              0
transfer_ovr_rank         88
transfer_pos_rank         35
transfer_stars             0
transfer_origin            5
transfer_destination      68
hs_class                   1
hs_rating_247           1012
hs_pos                     0
composite_rating        1121
composite_natl_rank     1123
composite_pos_rank      1123
source_hs_url              0
hs_stars                1012
source_player_url          0
transfer_status         2862
dtype: int64

In [ ]:
# display(portal_df_test[portal_df_test['transfer_destination'].isna()].head())
# print(portal_df_test[portal_df_test['transfer_destination'].isna()].iloc[-2]['source_player_url'])
# print(portal_df_test[portal_df_test['transfer_destination'].isna()].iloc[-2])
# # 46137152 dropped out

# display(portal_df_test[portal_df_test['hs_name'].isna()].head(11))
# print(portal_df_test[portal_df_test['hs_name'].isna()].iloc[-2]['source_player_url'])
# print(portal_df_test[portal_df_test['hs_name'].isna()].iloc[-2])
# # https://247sports.com/player/louis-brown-iv-46111905/college-310862 is a problem
# # https://247sports.com/player/easton-messer-46103582/college-286927/

# display(portal_df_test[portal_df_test['transfer_rating'].isna()].head(11))
# print(portal_df_test[portal_df_test['transfer_rating'].isna()].iloc[2]['source_player_url'])
# print(portal_df_test[portal_df_test['transfer_rating'].isna()].iloc[2])
# fine

# display(portal_df_test[portal_df_test['transfer_ovr_rank'].isna() &
#                        portal_df_test['transfer_rating'].notna()].head(11))
# print(portal_df_test[portal_df_test['transfer_ovr_rank'].isna() &
#                        portal_df_test['transfer_rating'].notna()].iloc[4]['source_player_url'])
# print(portal_df_test[portal_df_test['transfer_ovr_rank'].isna() &
#                        portal_df_test['transfer_rating'].notna()].iloc[4])
# fine


# display(portal_df_test[portal_df_test['pos_247'].isna()].head(11))
# print(portal_df_test[portal_df_test['pos_247'].isna()].iloc[-3]['source_player_url'])
# print(portal_df_test[portal_df_test['pos_247'].isna()].iloc[-3])

In [ ]:
# from src.utils.tests import test_timeline
# test_timeline(driver,
#               'https://247sports.com/player/jackson-ford-46132876/college-318734')

In [ ]:
# from src.scraper.player_scraper import scrape_player

# p = scrape_player(driver,
#                    'https://247sports.com/player/dylan-gooden-46116182/college-298901',)
# p

In [ ]:
trouble_links = ['https://247sports.com/player/jackson-ford-46132876/college-318734',
 'https://247sports.com/player/aidan-glover-46131125/college-311282',
 'https://247sports.com/player/jj-harrell-46136887/college-308341',
 'https://247sports.com/player/cayman-spaulding-46154253/college-326362',
 'https://247sports.com/player/greg-johnson-ii-46117160/college-336698']

In [ ]:
display(portal_df_test[portal_df_test['transfer_origin'].isna()][
    ['name', 'pos_247', 'source_player_url', 'transfer_destination', 'transfer_origin', 'transfer_status']])

,name,pos_247,source_player_url,transfer_destination,transfer_origin,transfer_status
2997,Jackson Ford,TE,https://247sports.com/player/jackson-ford-4613...,NaN,NaN,none
3001,Aidan Glover,QB,https://247sports.com/player/aidan-glover-4613...,NaN,NaN,none
3002,JJ Harrell,WR,https://247sports.com/player/jj-harrell-461368...,NaN,NaN,none
3004,Cayman Spaulding,LB,https://247sports.com/player/cayman-spaulding-...,NaN,NaN,none
3006,Greg Johnson II,CB,https://247sports.com/player/greg-johnson-ii-4...,NaN,NaN,none


In [ ]:
display(portal_df_test[portal_df_test['transfer_origin'].isna()]['source_player_url'].tolist())

['https://247sports.com/player/jackson-ford-46132876/college-318734',
 'https://247sports.com/player/aidan-glover-46131125/college-311282',
 'https://247sports.com/player/jj-harrell-46136887/college-308341',
 'https://247sports.com/player/cayman-spaulding-46154253/college-326362',
 'https://247sports.com/player/greg-johnson-ii-46117160/college-336698']

## Create '_category' Columns

In [5]:
teams_2025 = {
    "P4": {
        "ACC": [
            "Boston College", "California", "Clemson", "Duke", "Florida State",
            "Georgia Tech", "Louisville", "Miami", "NC State", "North Carolina",
            "Pittsburgh", "SMU", "Stanford", "Syracuse", "Virginia",
            "Virginia Tech", "Wake Forest",
        ],
        "SEC": [
            "Alabama", "Arkansas", "Auburn", "Florida", "Georgia", "Kentucky",
            "LSU", "Mississippi State", "Missouri", "Oklahoma", "Ole Miss",
            "South Carolina", "Tennessee", "Texas", "Texas A&M", "Vanderbilt",
        ],
        "B10": [
            "Illinois", "Indiana", "Iowa", "Maryland", "Michigan",
            "Michigan State", "Minnesota", "Nebraska", "Northwestern",
            "Ohio State", "Oregon", "Penn State", "Purdue", "Rutgers",
            "UCLA", "USC", "Washington", "Wisconsin",
        ],
        "B12": [
            "Arizona", "Arizona State", "Baylor", "BYU", "Cincinnati",
            "Colorado", "Houston", "Iowa State", "Kansas", "Kansas State",
            "Oklahoma State", "TCU", "Texas Tech", "UCF", "Utah",
            "West Virginia",
        ],
        "Independent": [
            "Notre Dame",
        ],
    },

    "G6": {
        "AAC": [
            "Army", "Charlotte", "East Carolina", "Florida Atlantic",
            "Memphis", "Navy", "North Texas", "Rice", "South Florida",
            "Temple", "Tulane", "Tulsa", "UAB", "UTSA",
        ],
        "CUSA": [
            # All are FBS for the 2025 season
            "Delaware", "FIU", "Jacksonville State", "Kennesaw State",
            "Liberty", "Louisiana Tech", "Middle Tennessee",
            "Missouri State", "New Mexico State", "Sam Houston",
            "UTEP", "Western Kentucky",
        ],
        "MW": [
            "Air Force", "Boise State", "Colorado State", "Fresno State",
            "Hawai'i", "Nevada", "New Mexico", "San Diego State",
            "San José State", "UNLV", "Utah State", "Wyoming",
        ],
        "MAC": [
            "Akron", "Ball State", "Bowling Green", "Buffalo",
            "Central Michigan", "Eastern Michigan", "Kent State",
            "Massachusetts", "Miami (OH)", "Northern Illinois",
            "Ohio", "Toledo", "Western Michigan",
        ],
        "SBC": [
            "App State", "Arkansas State", "Coastal Carolina",
            "Georgia Southern", "Georgia State", "James Madison",
            "Louisiana", "Marshall", "Old Dominion", "South Alabama",
            "Southern Miss", "Texas State", "Troy", "UL Monroe",
        ],
        # Pac-12 football in 2025
        "P12": [
            "Oregon State", "Washington State",
        ],
        "Independent": [
            "UConn",
        ],
    },

    "FCS": {
        # Any team whose FBS move is effective in 2026 or later stays here
        None: []  # populate programmatically or manually if needed
    },
}


In [6]:
portal_df_test.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3008 entries, 0 to 3007
Data columns (total 23 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   id_247                3008 non-null   int64  
 1   name                  3008 non-null   object 
 2   pos_247               3008 non-null   object 
 3   hs_name               3003 non-null   object 
 4   hs_city               3008 non-null   object 
 5   hs_state              3008 non-null   object 
 6   transfer_rating       3008 non-null   int64  
 7   transfer_year         3008 non-null   int64  
 8   transfer_ovr_rank     2920 non-null   float64
 9   transfer_pos_rank     2973 non-null   float64
 10  transfer_stars        3008 non-null   int64  
 11  transfer_origin       3003 non-null   object 
 12  transfer_destination  2940 non-null   object 
 13  hs_class              3007 non-null   float64
 14  hs_rating_247         1996 non-null   float64
 15  hs_pos               

In [7]:
def categorize_team(team_name, teams_dict):
    """Categorizes a team based on the provided dictionary."""
    for category, conferences in teams_dict.items():
        for conference, schools in conferences.items():
            if team_name in schools:
                return category
    return 'FCS'  # Default to FCS if no match is found

In [8]:
# Apply the categorization function to create new columns
portal_df_test['origin_category'] = portal_df_test['transfer_origin'].apply(categorize_team, args=(teams_2025,))
portal_df_test['destination_category'] = portal_df_test['transfer_destination'].apply(categorize_team, args=(teams_2025,))

## Excel Fill

### qwen 12/23 code

In [9]:
import re
import pandas as pd
from rapidfuzz import process, fuzz
from openpyxl import load_workbook

# ---------- helpers ----------
def build_portal_lookup(portal_df: pd.DataFrame):
    pdf = portal_df.copy()

    # keep "best" row per name (highest transfer_rating, then hs_rating_247)
    pdf["_sort_transfer"] = pdf["transfer_rating"].fillna(-1)
    pdf["_sort_hs"] = pdf.get("hs_rating_247", pd.Series([None]*len(pdf))).fillna(-1)
    pdf = pdf.sort_values(["name", "_sort_transfer", "_sort_hs"], ascending=[True, False, False])

    exact = dict(zip(pdf["name"], pdf.index))
    choices = pdf["name"].tolist()
    return pdf, exact, choices

def portal_to_fu_fields(row: pd.Series) -> dict:
    hs_city_state = None
    if pd.notna(row.get("hs_city")) and pd.notna(row.get("hs_state")):
        hs_city_state = f"{row.get('hs_city')}, {row.get('hs_state')}"
    hs_combo = None
    if pd.notna(row.get("hs_name")) and hs_city_state:
        hs_combo = f"{row.get('hs_name')} ({hs_city_state})"
    elif pd.notna(row.get("hs_name")):
        hs_combo = str(row.get("hs_name"))

    return {
        "247/On3 Position": row.get("pos_247"),
        "High School, City, State": hs_combo,
        "247 'Exp' (aka High School Class)": int(row["hs_class"]) if 
pd.notna(row.get("hs_class")) else None,
        "H/S Stars": int(row["hs_stars"]) if pd.notna(row.get("hs_stars")) else None,
        "H/S Rating": row.get("hs_rating_247"),
        "H/S National Rank": row.get("composite_natl_rank"),
        "H/S Position Rank": row.get("composite_pos_rank"),
        "Transfer Year": int(row["transfer_year"]) if pd.notna(row.get("transfer_year")) else 
None,
        "Transfer Origin": row.get("transfer_origin"),
        "Origin P4 / G5 / Non-FBS": row.get("origin_category"),
        "Transfer Destination": row.get("transfer_destination"),
        "Destination P4 / G5 / Non-FBS": row.get("destination_category"),
        "Transfer Stars": int(row["transfer_stars"]) if pd.notna(row.get("transfer_stars")) else 
None,
        "Transfer Rating": row.get("transfer_rating"),
        "Transfer Overall Rank": row.get("transfer_ovr_rank"),
        "Transfer Position Rank": row.get("transfer_pos_rank"),
    }

def fuzzy_match_one(name_raw: str, exact_map, choices, score_cutoff=95):
    if not name_raw:
        return None, 0
    if name_raw in exact_map:
        return exact_map[name_raw], 100

    match = process.extractOne(
        query=name_raw,
        choices=choices,
        scorer=fuzz.WRatio,          # changed from token_sort_ratio
        score_cutoff=score_cutoff
    )
    
    if match is None:
        return None, 0
    else:
        matched_norm, score, _ = match
        return exact_map.get(matched_norm), score

def fill_fu_columns(
    xlsx_in: str,
    xlsx_out: str,
    portal_df: pd.DataFrame,
    sheet_name: str,
    db_name_col: str = "Name",
    score_cutoff: int = 95,
    snaps_col: str | None = None,     # set to "Snaps" if you have it, else leave None
    snaps_min: int = 100
):
    # --- diagnostics: portal duplicates by normalized name ---
    tmp = portal_df.copy()
    dup_count = tmp.duplicated("name", keep=False).sum()
    print(f"Portal rows: {len(tmp)} | duplicate name rows: {dup_count}")

    pdf, exact_map, choices = build_portal_lookup(portal_df)

    wb = load_workbook(xlsx_in)
    ws = wb[sheet_name]

    # headers
    header_row = 1
    headers = {}
    for col in range(1, ws.max_column + 1):
        v = ws.cell(row=header_row, column=col).value
        if v is not None:
            headers[str(v).strip()] = col

    required = [
        db_name_col,
        "247/On3 Position",
        "High School, City, State",
        "247 'Exp' (aka High School Class)",
        "H/S Stars",
        "H/S Rating",
        "H/S National Rank",
        "H/S Position Rank",
        "Transfer Year",
        "Transfer Origin",
        "Origin P4 / G5 / Non-FBS",
        "Transfer Destination",
        "Destination P4 / G5 / Non-FBS",
        "Transfer Stars",
        "Transfer Rating",
        "Transfer Overall Rank",
        "Transfer Position Rank",
    ]
    missing = [c for c in required if c not in headers]
    if missing:
        raise ValueError(f"Missing these headers in the sheet: {missing}")

    if snaps_col is not None and snaps_col not in headers:
        raise ValueError(f"snaps_col='{snaps_col}' not found in sheet headers")

    name_col_idx = headers[db_name_col]
    transfer_year_col = headers["Transfer Year"]

    # --- NEW: enforce one-to-one matching (no portal id reused) ---
    used_portal_ids = set()

    matched = 0
    unmatched = 0
    skipped_filled = 0
    skipped_snaps = 0
    skipped_reuse = 0
    below_cutoff = 0

    for r in range(header_row + 1, ws.max_row + 1):
        # --- NEW: GATE: only fill if Transfer Year is blank (prevents mass filling non-transfer rows) ---
        ty = ws.cell(row=r, column=transfer_year_col).value
        if ty not in (None, ""):
            skipped_filled += 1
            continue

        # --- optional GATE: snaps >= 100 ---
        if snaps_col is not None:
            sv = ws.cell(row=r, column=headers[snaps_col]).value
            try:
                s = float(sv) if sv not in (None, "") else 0.0
            except Exception:
                s = 0.0
            if s < snaps_min:
                skipped_snaps += 1
                continue


        name_raw = ws.cell(row=r, column=name_col_idx).value
        portal_idx, score = fuzzy_match_one(name_raw, exact_map, choices, score_cutoff=score_cutoff)

        if portal_idx is None:
            unmatched += 1
            # print(f"Unmatched: {name_raw} (Score: {score})")  # Add debug print for unmatched rows
            continue
        if score < score_cutoff:
            below_cutoff += 1
            print(f"Below cutoff: {name_raw} (Score: {score})")  # Add debug print for rows below cutoff
            continue


        pid = int(pdf.loc[portal_idx, "id_247"])
        if pid in used_portal_ids:
            skipped_reuse += 1
            continue
        used_portal_ids.add(pid)

        prow = pdf.loc[portal_idx]
        vals = portal_to_fu_fields(prow)

        for k, v in vals.items():
            ws.cell(row=r, column=headers[k]).value = v

        matched += 1

    wb.save(xlsx_out)

    print(f"Saved: {xlsx_out}")
    print(f"Matched (written): {matched}")
    print(f"Unmatched: {unmatched}")
    print(f"Skipped (already had Transfer Year): {skipped_filled}")
    print(f"Skipped (snaps<{snaps_min}): {skipped_snaps}")
    print(f"Skipped (portal id reuse): {skipped_reuse}")
    print(f"Below cutoff (should be 0): {below_cutoff}")
    print(f"Unique portal ids used: {len(used_portal_ids)} (<= {len(portal_df)})")

    return unmatched

# ---------- RUN ----------
xlsx_in  = "data/2024-2025 Player Database v2.xlsx"
xlsx_out = "data/2024-2025 Player Database v2_FILLED_attempt2.xlsx"

unmatched = fill_fu_columns(
    xlsx_in=xlsx_in,
    xlsx_out=xlsx_out,
    portal_df=portal_df_test,      # your 3008-row df already in memory
    sheet_name="2024-2025 Player Database CSV",          # change
    db_name_col="full_name",           # change if your header differs
    score_cutoff=95,
    snaps_col='pff_snaps'           # <- set to "Snaps" if your sheet has it and you want the 100+ snaps gate
)


Portal rows: 3008 | duplicate name rows: 187
Saved: data/2024-2025 Player Database v2_FILLED_attempt2.xlsx
Matched (written): 2001
Unmatched: 8684
Skipped (already had Transfer Year): 0
Skipped (snaps<100): 20203
Skipped (portal id reuse): 885
Below cutoff (should be 0): 0
Unique portal ids used: 2001 (<= 3008)


In [ ]:
# --- diagnostics: portal duplicates by name ---
portal_df_test[portal_df_test.duplicated(subset="name", keep=False)]
portal_df_test['id_247'].nunique()

# example
portal_df_test[portal_df_test['name'] == 'Micah Hudson'].iloc[0]
portal_df_test[portal_df_test['name'] == 'Micah Hudson'].iloc[1]

,id_247,name,pos_247,hs_name,hs_city,hs_state,transfer_rating,transfer_year,transfer_ovr_rank,transfer_pos_rank,...,hs_pos,composite_rating,composite_natl_rank,composite_pos_rank,source_hs_url,hs_stars,source_player_url,transfer_status,origin_category,destination_category
61,46114316,Micah Hudson,WR,Lake Belton,Temple,TX,92,2025,64.0,15.0,...,WR,0.9934,16.0,4.0,https://247sports.com/player/micah-hudson-4611...,5.0,https://247sports.com/player/micah-hudson-4611...,NaN,P4,P4
67,46097332,Tanner Koziol,TE,Nazareth Academy,La Grange Park,IL,92,2025,72.0,3.0,...,WR,0.7859,2603.0,323.0,https://247sports.com/player/tanner-koziol-460...,2.0,https://247sports.com/player/tanner-koziol-460...,NaN,P4,P4
69,46100114,Julian Neal,CB,Mission,San Francisco,CA,92,2025,73.0,4.0,...,ATH,0.8348,1608.0,128.0,https://247sports.com/player/julian-neal-46100...,3.0,https://247sports.com/player/julian-neal-46100...,NaN,G6,P4
84,46135712,Nate Johnson,EDGE,Gaffney,Gaffney,SC,92,2025,89.0,12.0,...,EDGE,0.8470,1515.0,130.0,https://247sports.com/player/nate-johnson-4613...,3.0,https://247sports.com/player/nate-johnson-4613...,NaN,FCS,P4
92,46099592,Emmanuel Karnley,CB,Las Lomas,Walnut Creek,CA,91,2025,97.0,13.0,...,CB,0.8706,835.0,84.0,https://247sports.com/player/emmanuel-karnley-...,3.0,https://247sports.com/player/emmanuel-karnley-...,NaN,P4,P4
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2857,46156289,Robbie Pizzolato,IOL,John Curtis,New Orleans,LA,84,2025,NaN,NaN,...,IOL,NaN,NaN,NaN,https://247sports.com/player/robbie-pizzolato-...,NaN,https://247sports.com/player/robbie-pizzolato-...,NaN,FCS,G6
2858,46053702,Brandon Buckhaulter,WR,Hartfield Academy,Flowood,MS,83,2025,NaN,83.0,...,WR,0.8785,535.0,84.0,https://247sports.com/player/brandon-buckhault...,3.0,https://247sports.com/player/brandon-buckhault...,NaN,G6,FCS
2860,46042188,Jordon Simmons,RB,McEachern,Powder Springs,GA,86,2025,NaN,53.0,...,RB,0.8748,605.0,43.0,https://247sports.com/player/jordon-simmons-46...,3.0,https://247sports.com/player/jordon-simmons-46...,NaN,P4,G6
2921,46054109,Jayden Williams,CB,Centennial,Corona,CA,83,2025,2180.0,240.0,...,S,0.8500,1194.0,102.0,https://247sports.com/player/jayden-williams-4...,3.0,https://247sports.com/player/jayden-williams-4...,portal_only,G6,FCS


## Addn. Content

### Wikipedia Pull FCS Teams

In [ ]:
def populate_fcs_teams_from_wikipedia(d: dict) -> dict:
    """
    Pulls the 2025-season FCS program list and fills d["FCS"][None] with team/school names.
    Source table: https://en.wikipedia.org/wiki/List_of_NCAA_Division_I_FCS_football_programs
    """
    import pandas as pd

    url = "https://en.wikipedia.org/wiki/List_of_NCAA_Division_I_FCS_football_programs"
    tables = pd.read_html(url)

    # Find the table that contains the FCS programs list (has a 'Team' column).
    fcs_table = None
    for t in tables:
        if "Team" in t.columns:
            fcs_table = t
            break
    if fcs_table is None:
        raise RuntimeError("Couldn't find FCS programs table on Wikipedia page.")

    fcs_teams = (
        fcs_table["Team"]
        .astype(str)
        .str.replace(r"\s+\[.*\]$", "", regex=True)  # strip trailing footnote markers if present
        .str.strip()
        .tolist()
    )

    # Fill your schema
    d["FCS"][None] = fcs_teams
    return d

# If you want to populate immediately (requires internet):
# teams = populate_fcs_teams_from_wikipedia(teams)
